# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is best framed as a ranking/scoring task. The practical question is not just “is this page risky?” but “which pages or page groups should an SEO/content team review first?” I still need a classification-style target behind the score, because the score should learn from observed decline risk. For now, the starter CSV lets me practice at the page level. Later, the fuller project can move closer to my w01 idea by adding query overlap or graph features so the unit can become a page pair or cluster.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

possible_paths = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
]
DATA_PATH = next(path for path in possible_paths if path.exists())

df = pd.read_csv(DATA_PATH)
df['is_declining_proxy'] = df['trend_direction'].eq('down')

base_rate = df['is_declining_proxy'].mean()
print(f"Rows in starter slice: {len(df):,}")
print(f"Pseudonymous clients: {df['client_id'].nunique():,}")
print(f"Observed decline proxy rate: {base_rate:.1%}")
print("Task framing: score/rank pages for review, trained against an observed decline proxy.")


Rows in starter slice: 30,000
Pseudonymous clients: 32
Observed decline proxy rate: 54.2%
Task framing: score/rank pages for review, trained against an observed decline proxy.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target for the early notebook work is a decline-risk proxy: `is_declining_proxy = trend_direction == "down"`. This is an observed label in the starter dataset, but it is still only a proxy because the notebook has a fixed 90-day snapshot rather than a true future window. I should not use `trend_direction` or `trend_pct` as features, because those columns define the label. For the final lane, the cleaner target would be future performance loss for a page, pair, or cluster after a feature window, especially when several pages appear to compete for similar search demand.


In [2]:
label_counts = df['trend_direction'].value_counts().rename_axis('trend_direction').reset_index(name='rows')
print(label_counts.to_string(index=False))
print()
print(f"Declining rows: {df['is_declining_proxy'].sum():,} of {len(df):,}")
print("Leakage guard: trend_direction and trend_pct define the label, so they must not be features.")

safe_feature_examples = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr',
    'avg_position', 'content_age_days', 'days_since_last_update',
    'content_type', 'main_intent'
]
print("Example feature candidates to test carefully:")
for col in safe_feature_examples:
    print(f"- {col}")


trend_direction  rows
           down 16262
         stable  5962
             up  4388
            new  2236
           flat  1152

Declining rows: 16,262 of 30,000
Leakage guard: trend_direction and trend_pct define the label, so they must not be features.
Example feature candidates to test carefully:
- impressions_90d
- clicks_90d
- sessions_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update
- content_type
- main_intent


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main success metric should be precision@K, especially precision@50, because the real user is a busy SEO/content team that can only review a limited queue. A useful system should make the top of the queue much richer in observed decline/cannibalization risk than the general page pool. For this starter framing, I would call it useful if the top 50 reviewed pages are at least about 10 percentage points above the base decline rate, while still passing a client-holdout check later.


In [3]:
K = 50
minimum_lift = 0.10
target_precision_at_k = base_rate + minimum_lift

print(f"Base decline proxy rate: {base_rate:.1%}")
print(f"Review budget K: {K} pages")
print(f"Defensible starter target: precision@{K} >= {target_precision_at_k:.1%}")
print("Later validation should use client-holdout or time-based splits, not random rows only.")


Base decline proxy rate: 54.2%
Review budget K: 50 pages
Defensible starter target: precision@50 >= 64.2%
Later validation should use client-holdout or time-based splits, not random rows only.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In the starter CSV, one row is one pseudonymous content item/page. For the final cannibalization idea, I may transform this into one row per page pair or page cluster after adding query-overlap and graph features from the warehouse data. For this notebook, I will stay honest and show the page-level framing that the current data can actually support.


In [4]:
lane_df = df[[
    'content_id', 'client_id', 'content_type', 'main_intent',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr',
    'avg_position', 'days_since_last_update', 'is_declining_proxy'
]].copy()

print(f"Unit of analysis now: one row = one content item/page")
print(f"Rows: {len(lane_df):,}")
print(f"Columns shown for framing: {lane_df.shape[1]}")
display(lane_df.head(8))


Unit of analysis now: one row = one content item/page
Rows: 30,000
Columns shown for framing: 11


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,days_since_last_update,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,0.76,10.6,20,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,0.05,20.3,25,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,0.09,36.5,20,True
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,0.49,6.2,22,False
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,0.13,44.0,14,True
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,0.03,8.5,20,True
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,0.00,7.0,20,True
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,0.06,21.2,22,False


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can be a baseline, but it is probably too narrow for this lane. Cannibalization risk can depend on demand, CTR, position, freshness, content type, engagement, client-level differences, and later query overlap between pages. These signals can disagree with each other. A rule such as “high impressions, low CTR, and stale” may find some useful cases, but it will miss pages where the risk comes from a different combination of signals. ML is useful only if it improves the ranked review queue against this simple-rule baseline.


In [5]:
high_demand = df['impressions_90d'] >= df['impressions_90d'].quantile(0.75)
low_ctr = df['ctr'] <= df['ctr'].quantile(0.25)
stale = df['days_since_last_update'] > 30
fixed_rule = np.logical_and.reduce([high_demand, low_ctr, stale])

measurable = np.logical_and(df['impressions_90d'] >= 100, df['sessions_90d'] > 0)

print(f"Base decline proxy rate: {base_rate:.1%}")
print(f"Pages with measurable opportunity: {measurable.sum():,} ({measurable.mean():.1%})")
print(f"Fixed-rule pages found: {fixed_rule.sum():,} ({fixed_rule.mean():.1%})")
print(f"Fixed-rule decline proxy rate: {df.loc[fixed_rule, 'is_declining_proxy'].mean():.1%}")
print()
print("Why a learned ranking may help:")
print(f"- content types: {df['content_type'].nunique()} types")
print(f"- word_count missing: {df['word_count'].isna().mean():.1%}")
print(f"- search_volume missing: {df['search_volume'].isna().mean():.1%}")
print("- later warehouse work can add query-overlap/graph features for the cannibalization idea")


Base decline proxy rate: 54.2%
Pages with measurable opportunity: 22,006 (73.4%)
Fixed-rule pages found: 74 (0.2%)
Fixed-rule decline proxy rate: 75.7%

Why a learned ranking may help:
- content types: 3 types
- word_count missing: 25.7%
- search_volume missing: 8.2%
- later warehouse work can add query-overlap/graph features for the cannibalization idea


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
